In [2]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss
from sklearn.base import is_classifier
from dataclasses import dataclass
from typing import List, Iterator

DATA_NPZ_PATH = "../../datasets/wine_preprocessed.npz"
OUT_DIR = "./outputs_wine"
os.makedirs(OUT_DIR, exist_ok=True)

N_TEST_SAMPLES = 10
SAMPLE_SEED = 123
RANDOM_STATE = 42

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from cgt_xai.models import get_model
from cgt_xai.concepts.shapley import exact_shapley_value
from cgt_xai.concepts.least_core import exact_least_core
from cgt_xai.concepts.nucleolus import exact_nucleolus
from cgt_xai.concepts.gately_point import exact_gately_value
from cgt_xai.concepts.banzhaf import exact_banzhaf_index

def load_strict_data(npz_path):
    if not os.path.exists(npz_path):
        raise FileNotFoundError(f"NPZ not found at: {npz_path}")

    print(f"Loading data from {npz_path}...")
    d = np.load(npz_path, allow_pickle=True)

    X_train = d["X_train"].astype(float)
    X_test  = d["X_test"].astype(float)
    y_train = d["y_train"].astype(int).ravel()
    y_test  = d["y_test"].astype(int).ravel()

    if "feature_names" in d:
        feature_names = d["feature_names"].tolist()
    else:
        feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    print(f"Shapes -> X_train: {X_train.shape}, y_train: {y_train.shape}, X_test: {X_test.shape}")
    return X_train, y_train, X_test, y_test, feature_names

X_train, y_train, X_test, y_test, feature_names = load_strict_data(DATA_NPZ_PATH)
n_features = X_train.shape[1]
all_labels = sorted(np.unique(y_train).tolist())
baseline_full = np.mean(X_train, axis=0)

trained_models = {}
MODEL_NAMES = ["logistic", "svm", "random_forest"]

print("\nTraining models...")
for m_name in MODEL_NAMES:
    model = get_model(m_name, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    trained_models[m_name] = model

@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]
    def __len__(self) -> int: return len(self.indices)
    def __iter__(self) -> Iterator[int]: return iter(self.indices)
    def __getitem__(self, i: int) -> int: return self.indices[i]

class LocalLossUtility:
    def __init__(self, model, x_full, y_true, baseline_full, feature_subset, feature_names, all_labels):
        self.model = model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.feature_subset = np.asarray(feature_subset, dtype=int).ravel()
        self.y_true = int(y_true)
        self.all_labels = all_labels

        subset_names = [str(feature_names[j]) for j in feature_subset]
        self.data = FeaturePlayers(
            indices=list(range(len(feature_subset))),
            data_names=subset_names
        )

    def _x_for_coalition(self, S):
        x = self.baseline_full.copy()
        for local_j in S:
            global_j = self.feature_subset[int(local_j)]
            x[global_j] = self.x_full[global_j]
        return x.reshape(1, -1)

    def _loss_value(self, X_in) -> float:
        if hasattr(self.model, "predict_proba"):
            proba = self.model.predict_proba(X_in)
            return float(log_loss([self.y_true], proba, labels=self.all_labels))
        else:
            pred = self.model.predict(X_in)[0]
            return 0.0 if pred == self.y_true else 1.0

    def __call__(self, S):
        return -self._loss_value(self._x_for_coalition(S))

rng = np.random.default_rng(SAMPLE_SEED)
test_idx = rng.choice(len(X_test), size=min(N_TEST_SAMPLES, len(X_test)), replace=False)

METHODS = [
    ("shapley", exact_shapley_value),
    ("least_core", exact_least_core),
    ("gately", exact_gately_value),
    ("banzhaf", exact_banzhaf_index),
]

records = []
print(f"\nRunning explanation on {len(test_idx)} samples...")

for model_name, model in trained_models.items():
    for i, idx in enumerate(test_idx):
        x = X_test[idx]
        y = y_test[idx]
        feat_subset = np.arange(n_features)

        util = LocalLossUtility(
            model=model, x_full=x, y_true=y, baseline_full=baseline_full,
            feature_subset=feat_subset, feature_names=feature_names, all_labels=all_labels
        )

        for method_name, method_fn in METHODS:
            try:
                res = method_fn(util)
                alloc = getattr(res, 'values', res)
                alloc = np.asarray(alloc, dtype=float).ravel()

                records.append({
                    "model": model_name,
                    "test_index": int(idx),
                    "y_true": int(y),
                    "method": method_name,
                    "alloc_full": alloc.tolist()
                })
            except Exception as e:
                print(f"Error in {method_name}: {e}")

df = pd.DataFrame(records)
df.to_csv(os.path.join(OUT_DIR, "loss_utility_results.csv"), index=False)
print("Done.")

Loading data from ../datasets/wine_preprocessed.npz...
Shapes -> X_train: (141, 13), y_train: (141,), X_test: (36, 13)

Training models...

Running explanation on 10 samples...
Done.


In [ ]:
import os
import sys
import warnings
import logging
import ast
from pathlib import Path
from dataclasses import dataclass
from typing import List

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import quantus
from sklearn.preprocessing import LabelEncoder

# Setup Paths
sys.path.append(os.path.abspath(os.path.join("..", "..")))

# Import GT methods
from cgt_xai.concepts.shapley import exact_shapley_value
from cgt_xai.concepts.least_core import exact_least_core
from cgt_xai.concepts.gately_point import exact_gately_value
from cgt_xai.concepts.banzhaf import exact_banzhaf_index
from cgt_xai.models import get_model

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Configuration
DATA_NPZ_PATH = "../../datasets/wine_preprocessed.npz"
RESULTS_CSV_PATH = "./outputs_wine/loss_utility_results.csv"
OUT_DIR = Path("./quantus_wine/quantus_evaluation")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV_PATH = OUT_DIR / "quantus_results_incremental.csv"

RANDOM_STATE = 42
AOPC_TOPK_LIST = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]

# 1. utilities

@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]
    def __len__(self): return len(self.indices)
    def __iter__(self): return iter(self.indices)
    def __getitem__(self, i): return self.indices[i]

class LocalLossUtility:
    def __init__(self, model, x_full, y_true, baseline_full, feature_subset, feature_names, all_labels):
        self.model = model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.feature_subset = np.asarray(feature_subset, dtype=int).ravel()
        self.y_true = int(y_true)
        self.all_labels = all_labels

        subset_names = [str(feature_names[i]) for i in self.feature_subset]
        self.data = FeaturePlayers(indices=list(range(len(feature_subset))), data_names=subset_names)

    def _x_for_coalition(self, S):
        x = self.baseline_full.copy()
        for local_j in S:
            global_j = self.feature_subset[int(local_j)]
            x[global_j] = self.x_full[global_j]
        return x.reshape(1, -1)

    def _loss_value(self, X_in) -> float:
        from sklearn.metrics import log_loss
        if hasattr(self.model, "predict_proba"):
            proba = self.model.predict_proba(X_in)
            return float(log_loss([self.y_true], proba, labels=self.all_labels))
        else:
            pred = self.model.predict(X_in)[0]
            return 0.0 if pred == self.y_true else 1.0

    def __call__(self, S):
        return -self._loss_value(self._x_for_coalition(S))

class SklearnTorchWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.is_clf = hasattr(model, "predict_proba")

    def forward(self, x):
        if isinstance(x, torch.Tensor):
            x_np = x.detach().cpu().numpy()
        else:
            x_np = x

        if x_np.ndim == 3:
            x_np = x_np.reshape(x_np.shape[0], -1)
        elif x_np.ndim == 1:
            x_np = x_np.reshape(1, -1)

        if self.is_clf:
            out = self.model.predict_proba(x_np)
        else:
            out = self.model.predict(x_np)
            if out.ndim == 1: out = out.reshape(-1, 1)

        return torch.from_numpy(out).float()

def load_data_strict():
    d = np.load(DATA_NPZ_PATH, allow_pickle=True)
    X_train = d["X_train"].astype(float)
    X_test = d["X_test"].astype(float)
    y_train = d["y_train"].astype(int).ravel()
    y_test = d["y_test"].astype(int).ravel()

    if "feature_names" in d:
        feature_names = d["feature_names"].tolist()
    else:
        feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    return X_train, X_test, y_train, y_test, feature_names

# 2. robust perturbation

def perturb_tabular_robust(arr, indices, baseline, **kwargs):
    arr_pert = arr.copy()
    base_vec = baseline.flatten()

    if arr.ndim == 2:
        F = arr.shape[1]
    elif arr.ndim == 3:
        F = arr.shape[2]
    else:
        F = len(base_vec)

    def apply_mask(target_slice, idxs):
        valid_idxs = [i for i in idxs if i < F]
        if valid_idxs:
            target_slice[valid_idxs] = base_vec[valid_idxs]

    B = arr.shape[0]
    for b in range(B):
        idx_list = indices[b] if isinstance(indices[b], (list, np.ndarray)) else [indices[b]]
        if arr.ndim == 2:
            apply_mask(arr_pert[b], idx_list)
        elif arr.ndim == 3:
            apply_mask(arr_pert[b, 0], idx_list)

    return arr_pert

def perturb_uniform_robust(arr, **kwargs):
    lower = kwargs.get('lower_bound', 0.01)
    upper = kwargs.get('upper_bound', 0.02)
    noise = np.random.uniform(lower, upper, size=arr.shape)
    sign = np.random.choice([-1, 1], size=arr.shape)
    return arr + (noise * sign)

def distance_euclidean(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

# 3. metric definitions

def get_metrics(baseline_vec):
    base_3d = baseline_vec.reshape(1, 1, -1)
    perturb_kwargs = {"baseline": base_3d}

    metrics = {
        "faithfulness_corr": quantus.FaithfulnessCorrelation(
            nr_runs=10,
            subset_size=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "faithfulness_estimate": quantus.FaithfulnessEstimate(
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "selectivity": quantus.Selectivity(
            patch_size=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "monotonicity": quantus.Monotonicity(
            features_in_step=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "pixel_flipping": quantus.PixelFlipping(
            features_in_step=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "sparseness": quantus.Sparseness(return_aggregate=True),
        "complexity": quantus.Complexity(return_aggregate=True),

        "local_lipschitz_estimate": quantus.LocalLipschitzEstimate(
            nr_samples=1, perturb_std=0.1,
            norm_numerator=distance_euclidean,
            norm_denominator=distance_euclidean,
            perturb_func=perturb_uniform_robust,
            perturb_func_kwargs={},
            return_aggregate=True
        ),
        "max_sensitivity": quantus.MaxSensitivity(
            nr_samples=1, lower_bound=0.1, upper_bound=0.2,
            norm_numerator=quantus.norm_func.fro_norm,
            norm_denominator=quantus.norm_func.fro_norm,
            perturb_func=perturb_uniform_robust,
            perturb_func_kwargs={},
            return_aggregate=True
        )
    }
    return metrics

def compute_aopc_manual(sk_model, X_test_2d, y_test_indices, attributions_2d, top_k_list, baseline):
    if hasattr(sk_model, "predict_proba"):
        base_probs = sk_model.predict_proba(X_test_2d)
        # Use encoded indices to select columns
        base_scores = base_probs[np.arange(len(y_test_indices)), y_test_indices]
    else:
        return np.nan

    drops_all_k = []
    valid_k_list = [k for k in top_k_list if k <= X_test_2d.shape[1]]

    for k in valid_k_list:
        X_perturbed = X_test_2d.copy()
        for i in range(len(X_test_2d)):
            top_indices = np.argsort(-np.abs(attributions_2d[i]))[:k]
            X_perturbed[i, top_indices] = baseline[top_indices]

        new_probs = sk_model.predict_proba(X_perturbed)
        new_scores = new_probs[np.arange(len(y_test_indices)), y_test_indices]
        drops_all_k.append(np.mean(base_scores - new_scores))
    return np.mean(drops_all_k) if drops_all_k else np.nan

# 4. explain function

def make_explain_func(method_name, all_labels, baseline_full, feature_names):
    METHOD_MAP = {
        "shapley": exact_shapley_value,
        "least_core": exact_least_core,
        "gately": exact_gately_value,
        "banzhaf": exact_banzhaf_index
    }
    method_fn = METHOD_MAP.get(method_name)

    if method_fn is None:
        def dummy(model, inputs, targets, **kwargs): return np.zeros((len(inputs), 1, len(feature_names)))
        return dummy

    def explain_func(model, inputs, targets, **kwargs):
        if isinstance(model, SklearnTorchWrapper):
            real_model = model.model
        else:
            real_model = model

        attributions = []
        for i in range(len(inputs)):
            x_in = inputs[i]
            if hasattr(x_in, "detach"):
                x_in = x_in.detach().cpu().numpy()
            x = x_in.flatten()

            y_in = targets[i]
            y = int(y_in.item()) if hasattr(y_in, "item") else int(y_in)

            # because the model expects those specific labels for log_loss
            # BUT the index passed to Quantus must be 0,1,2
            # We need to map back to original label if needed.
            # Check model classes:
            real_label = real_model.classes_[y] if hasattr(real_model, "classes_") else y

            util = LocalLossUtility(
                model=real_model, x_full=x, y_true=real_label,
                baseline_full=baseline_full, feature_subset=np.arange(len(x)),
                feature_names=feature_names, all_labels=all_labels
            )

            try:
                res = method_fn(util)
                alloc = getattr(res, 'values', res)
                attributions.append(np.asarray(alloc, dtype=float).reshape(1, -1))
            except Exception as e:
                attributions.append(np.zeros((1, len(x))))

        return np.array(attributions)

    return explain_func

# 5. main

def main():
    # 1. Load Data
    X_train, X_test, y_train, y_test, feature_names = load_data_strict()
    baseline = np.mean(X_train, axis=0)
    all_labels = sorted(np.unique(y_train).tolist())

    le = LabelEncoder()
    le.fit(y_train)
    # We ONLY transform y_subset for Quantus indexing

    # 3. Load Results
    if not os.path.exists(RESULTS_CSV_PATH):
        print(f"File not found: {RESULTS_CSV_PATH}")
        return

    df_results = pd.read_csv(RESULTS_CSV_PATH)
    valid_indices = sorted(df_results["test_index"].unique())

    X_subset = X_test[valid_indices]
    y_subset_raw = y_test[valid_indices]

    # Transform labels to 0, 1, 2 for Quantus indexing
    y_subset_indices = le.transform(y_subset_raw)

    x_batch = X_subset[:, np.newaxis, :]

    models_to_run = df_results["model"].unique()
    methods_to_run = df_results["method"].unique()

    metrics = get_metrics(baseline)

    logger.info(f"Loaded {len(valid_indices)} samples.")

    # Define columns explicitly
    cols = ["model", "method", "faithfulness_corr", "faithfulness_estimate",
            "selectivity", "monotonicity", "pixel_flipping", "sparseness",
            "complexity", "local_lipschitz_estimate", "max_sensitivity", "aopc_manual"]

    # Check if file exists to determine if we need to write headers
    if not OUT_CSV_PATH.exists():
        pd.DataFrame(columns=cols).to_csv(OUT_CSV_PATH, index=False)
        processed_pairs = set()
    else:
        # Load existing progress to resume if needed
        existing_df = pd.read_csv(OUT_CSV_PATH)
        processed_pairs = set(zip(existing_df["model"], existing_df["method"]))
        logger.info(f"Resuming... Found {len(processed_pairs)} already processed entries.")

    for model_name in models_to_run:
        logger.info(f"--- Evaluating Model: {model_name} ---")

        # Train original model (on original labels 1,2,3)
        sk_model = get_model(model_name, random_state=RANDOM_STATE)
        sk_model.fit(X_train, y_train)
        torch_model = SklearnTorchWrapper(sk_model).eval()

        df_model = df_results[df_results["model"] == model_name]

        for method_name in methods_to_run:
            # Check if already processed
            if (model_name, method_name) in processed_pairs:
                logger.info(f"   Method: {method_name} (SKIPPING - ALREADY DONE)")
                continue

            logger.info(f"   Method: {method_name}")

            attributions_list = []
            for idx in valid_indices:
                row = df_model[(df_model["test_index"] == idx) & (df_model["method"] == method_name)]
                if row.empty:
                    attributions_list.append(np.zeros(X_train.shape[1]))
                else:
                    val_str = row.iloc[0]["alloc_full"]
                    try:
                        if isinstance(val_str, str):
                            val_arr = np.array(ast.literal_eval(val_str))
                        else:
                            val_arr = np.array(val_str)
                    except:
                        val_arr = np.zeros(X_train.shape[1])
                    attributions_list.append(val_arr)

            a_batch_2d = np.array(attributions_list)
            a_batch_3d = a_batch_2d[:, np.newaxis, :]

            row_stats = {"model": model_name, "method": method_name}

            # Pass y_subset_indices (0, 1, 2) here!
            for m_name, metric_fn in metrics.items():
                if m_name in ["local_lipschitz_estimate", "max_sensitivity"]: continue
                try:
                    scores = metric_fn(
                        model=torch_model,
                        x_batch=x_batch,
                        y_batch=y_subset_indices,
                        a_batch=a_batch_3d,
                        channel_first=True,
                        softmax=False
                    )
                    row_stats[m_name] = np.nanmean(scores)
                except Exception as e:
                    row_stats[m_name] = np.nan

            try:
                # Pass y_subset_indices here too
                aopc = compute_aopc_manual(sk_model, X_subset, y_subset_indices, a_batch_2d, AOPC_TOPK_LIST, baseline)
                row_stats["aopc_manual"] = aopc
            except Exception as e:
                 row_stats["aopc_manual"] = np.nan

            explain_func = make_explain_func(method_name, all_labels, baseline, feature_names)

            for m_name in ["local_lipschitz_estimate", "max_sensitivity"]:
                # SPEED OPTIMIZATION: Skip robustness for Random Forest to ensure completion
                if model_name == "random_forest":
                     row_stats[m_name] = np.nan
                     continue

                try:
                    scores = metrics[m_name](
                        model=torch_model,
                        x_batch=x_batch,
                        y_batch=y_subset_indices,
                        explain_func=explain_func,
                        channel_first=True,
                        softmax=False
                    )
                    row_stats[m_name] = np.nanmean(scores)
                except Exception as e:
                    row_stats[m_name] = np.nan

            f_corr = row_stats.get('faithfulness_corr', np.nan)
            ms = row_stats.get('max_sensitivity', np.nan)
            logger.info(f"     -> Faith_Corr: {f_corr:.4f}, MaxSens: {ms:.4f}")

            # Save strictly this row to the CSV
            df_row = pd.DataFrame([row_stats])
            # Reorder columns to match header
            df_row = df_row.reindex(columns=cols)
            df_row.to_csv(OUT_CSV_PATH, mode='a', header=False, index=False)
            logger.info(f"     [SAVED] {model_name} - {method_name}")

    print(f"\nFinal results saved incrementally to: {OUT_CSV_PATH}")

if __name__ == "__main__":
    main()


Final results saved incrementally to: quantus_wine/quantus_evaluation/quantus_results_incremental.csv


In [1]:
import os
import ast
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import log_loss

# Config
DATA_NPZ_PATH = "../../datasets/wine_preprocessed.npz"
RESULTS_CSV = "./outputs_wine/loss_utility_results.csv"
OUTPUT_DIR = "./outputs_wine/insertion_deletion_plots"

RANDOM_STATE = 42
N_STEPS = 20

os.makedirs(OUTPUT_DIR, exist_ok=True)

METHOD_NAME_MAP = {
    "shapley": "Shapley",
    "banzhaf": "Banzhaf",
    "gately": "Gately",
    "least_core": "Least Core",
    "nucleolus": "Nucleolus"
}

# Import project modules
repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from cgt_xai.models import get_model

# Load data
data = np.load(DATA_NPZ_PATH, allow_pickle=True)
X_train = data["X_train"].astype(float)
X_test  = data["X_test"].astype(float)
y_train = data["y_train"].astype(int).ravel()
y_test  = data["y_test"].astype(int).ravel()

if "feature_names" in data:
    feature_names = data["feature_names"].tolist()
else:
    feature_names = [f"f{i}" for i in range(X_train.shape[1])]

baseline_full = np.mean(X_train, axis=0)
all_labels = sorted(np.unique(y_train).tolist())

# Train models
MODEL_NAMES = sorted(pd.read_csv(RESULTS_CSV)["model"].unique().tolist())

trained_models = {}
for model_name in MODEL_NAMES:
    print(f"Training model: {model_name}")
    model = get_model(model_name, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    trained_models[model_name] = model

# Load attributions
df = pd.read_csv(RESULTS_CSV)

def parse_alloc(x):
    if isinstance(x, str):
        return np.array(ast.literal_eval(x), dtype=float)
    return np.array(x, dtype=float)

df["alloc_full_parsed"] = df["alloc_full"].apply(parse_alloc)

# Loss function
def compute_instance_loss(model, x_row, y_true):
    x_row = x_row.reshape(1, -1)

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(x_row)
        return float(log_loss([y_true], proba, labels=all_labels))
    else:
        pred = model.predict(x_row)[0]
        return 0.0 if pred == y_true else 1.0

# Curves
def deletion_curve_loss(model, x, y_true, attribution, baseline, n_steps=20):
    order = np.argsort(-np.abs(attribution))
    xs = np.linspace(0, 1, n_steps + 1)
    ys = []

    for frac in xs:
        k = int(round(frac * len(attribution)))
        x_mod = x.copy()
        x_mod[order[:k]] = baseline[order[:k]]
        ys.append(compute_instance_loss(model, x_mod, y_true))

    return xs, np.array(ys)

def insertion_curve_loss(model, x, y_true, attribution, baseline, n_steps=20):
    order = np.argsort(-np.abs(attribution))
    xs = np.linspace(0, 1, n_steps + 1)
    ys = []

    for frac in xs:
        k = int(round(frac * len(attribution)))
        x_mod = baseline.copy()
        x_mod[order[:k]] = x[order[:k]]
        ys.append(compute_instance_loss(model, x_mod, y_true))

    return xs, np.array(ys)

def curve_auc(x, y):
    return float(np.trapezoid(y, x))

# Main loop
summary_rows = []

for model_name in sorted(df["model"].unique()):
    print(f"\nProcessing model: {model_name}")
    model = trained_models[model_name]

    methods = sorted([m for m in df[df["model"] == model_name]["method"].unique()
    if m != "shapley_shubik"
])

    deletion_plot_data = {}
    insertion_plot_data = {}

    for method_name in methods:
        pretty_name = METHOD_NAME_MAP.get(method_name, method_name)

        sub = df[(df["model"] == model_name) & (df["method"] == method_name)]

        deletion_curves = []
        insertion_curves = []

        for _, row in sub.iterrows():
            idx = int(row["test_index"])
            y_true = int(row["y_true"])
            attribution = row["alloc_full_parsed"]

            x = X_test[idx]

            x_del, y_del = deletion_curve_loss(model, x, y_true, attribution, baseline_full, N_STEPS)
            x_ins, y_ins = insertion_curve_loss(model, x, y_true, attribution, baseline_full, N_STEPS)

            deletion_curves.append(y_del)
            insertion_curves.append(y_ins)

        deletion_curves = np.array(deletion_curves)
        insertion_curves = np.array(insertion_curves)

        del_mean = deletion_curves.mean(axis=0)
        del_std = deletion_curves.std(axis=0)

        # Normalize
        del_mean = (del_mean - del_mean[0]) / (del_mean[-1] - del_mean[0] + 1e-8)
        del_std = del_std / (del_mean[-1] - del_mean[0] + 1e-8)

        ins_mean = insertion_curves.mean(axis=0)
        ins_std = insertion_curves.std(axis=0)

        # Normalize
        ins_mean = (ins_mean - ins_mean[-1]) / (ins_mean[0] - ins_mean[-1] + 1e-8)
        ins_std = ins_std / (ins_mean[0] - ins_mean[-1] + 1e-8)

        del_auc = curve_auc(x_del, del_mean)
        ins_auc = curve_auc(x_ins, ins_mean)

        summary_rows.append({
            "model": model_name,
            "method": method_name,
            "deletion_auc_loss": del_auc,
            "insertion_auc_loss": ins_auc
        })

        deletion_plot_data[pretty_name] = (x_del, del_mean, del_std, del_auc)
        insertion_plot_data[pretty_name] = (x_ins, ins_mean, ins_std, ins_auc)

    plt.figure(figsize=(8, 5))
    for name, (x_vals, mean, std, auc_val) in deletion_plot_data.items():
        plt.plot(x_vals, mean, label=f"{name} (AUC={auc_val:.4f})")
        plt.fill_between(x_vals, mean - std, mean + std, alpha=0.15)

    plt.xlabel("Fraction of Features Perturbed")
    plt.ylabel("Loss")
    plt.title(f"Deletion Curves - {model_name}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/deletion_{model_name}.png", dpi=300)
    plt.close()

    plt.figure(figsize=(8, 5))
    for name, (x_vals, mean, std, auc_val) in insertion_plot_data.items():
        plt.plot(x_vals, mean, label=f"{name} (AUC={auc_val:.4f})")
        plt.fill_between(x_vals, mean - std, mean + std, alpha=0.15)

    plt.xlabel("Fraction of Features Restored")
    plt.ylabel("Loss")
    plt.title(f"Insertion Curves - {model_name}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/insertion_{model_name}.png", dpi=300)
    plt.close()

# Save summary
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f"{OUTPUT_DIR}/auc_summary.csv", index=False)

print("\nDone!")
print(summary_df)

Training model: logistic
Training model: random_forest
Training model: svm

Processing model: logistic

Processing model: random_forest

Processing model: svm

Done!
            model      method  deletion_auc_loss  insertion_auc_loss
0        logistic     banzhaf           0.822628            0.062136
1        logistic      gately           0.332902            0.206554
2        logistic  least_core           0.384597            0.192215
3        logistic     shapley           0.817922            0.062999
4   random_forest     banzhaf           0.859051            0.098430
5   random_forest      gately           0.641528            0.217307
6   random_forest  least_core           0.630486            0.210179
7   random_forest     shapley           0.860068            0.100647
8             svm     banzhaf           0.812316            0.050239
9             svm      gately           0.367141            0.205487
10            svm  least_core           0.369984            0.240988
11    